# Chapter 3 Practical 04: Item-Item Collaborative Filtering

Learning objectives:
- Compute item-item similarities from user ratings.
- Predict a missing rating from similar items the user already rated.
- Compare item-item CF with user-user CF.
- Discuss why item-item CF is often easier to cache and serve.

Slide connection: item-item CF concept, item-item prediction example, and scalability.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Loaded ratings_chapter3.csv from data/ratings_chapter3.csv
Loaded movies_chapter3.csv from data/movies_chapter3.csv


title,Blade Runner,Finding Nemo,Independence Day,Jurassic Park,Star Wars,Terminator 2,The Matrix,The Notebook,Titanic,Toy Story
user_id,,,,,,,,,,
Alice,5.0,NaN,4.0,NaN,4.0,NaN,5.0,NaN,NaN,NaN
Bob,NaN,NaN,6.0,4.0,7.0,4.0,7.0,NaN,NaN,NaN
Chris,NaN,NaN,2.0,7.0,3.0,7.0,NaN,NaN,NaN,5.0
Karen,NaN,NaN,NaN,4.0,7.0,3.0,6.0,NaN,NaN,NaN
Lynn,NaN,NaN,2.0,4.0,4.0,6.0,NaN,NaN,NaN,6.0
Nina,NaN,5.0,NaN,4.0,NaN,NaN,NaN,NaN,2.0,5.0
Omar,NaN,2.0,NaN,3.0,NaN,NaN,NaN,5.0,5.0,NaN
Sally,NaN,NaN,7.0,6.0,7.0,3.0,6.0,NaN,NaN,NaN


## Packages and key functions used

- `pandas` is used for tabular data. Important functions in this notebook include `read_csv`, `merge`, `pivot_table`, `groupby`, `agg`, `sort_values`, and `dropna`.
- `numpy` is used for numerical operations such as vector norms, dot products, averages, absolute values, and exponential time-decay weights.
- `pathlib.Path` helps the notebook find local CSV files in Colab, Jupyter, or the repository folder.
- The shared helper `read_chapter3_csv()` first looks for local data files and then falls back to the GitHub raw URL when students open the notebook directly online.
- `rating_matrix` is the central memory-based collaborative filtering object: rows are users, columns are movies, observed numbers are ratings, and missing cells mean unknown preferences.


## Technique: item-item similarity matrix

Item-item CF compares columns of the user-item matrix. Two movies are compared only using users who rated both movies.

Important function details:

- `matrix[[item_a, item_b]].dropna()` keeps only users who rated both items.
- `np.corrcoef()` calculates Pearson correlation between the two item columns.
- The nested loop fills a square item-item similarity matrix that can be reused for many users.


In [2]:
def item_pearson(matrix, item_a, item_b):
    pair = matrix[[item_a, item_b]].dropna()
    if len(pair) < 2:
        return np.nan
    if pair[item_a].std() == 0 or pair[item_b].std() == 0:
        return np.nan
    return float(np.corrcoef(pair[item_a], pair[item_b])[0, 1])

items = rating_matrix.columns
item_sim = pd.DataFrame(index=items, columns=items, dtype=float)
for a in items:
    for b in items:
        item_sim.loc[a, b] = 1.0 if a == b else item_pearson(rating_matrix, a, b)

item_sim.round(2)


title,Blade Runner,Finding Nemo,Independence Day,Jurassic Park,Star Wars,Terminator 2,The Matrix,The Notebook,Titanic,Toy Story
title,,,,,,,,,,
Blade Runner,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Finding Nemo,NaN,1.0,NaN,1.00,NaN,NaN,NaN,NaN,-1.0,NaN
Independence Day,NaN,NaN,1.00,-0.11,0.94,-0.97,0.65,NaN,NaN,NaN
Jurassic Park,NaN,1.0,-0.11,1.00,-0.45,0.39,-0.50,NaN,-1.0,-0.5
Star Wars,NaN,NaN,0.94,-0.45,1.00,-0.97,0.82,NaN,NaN,1.0
Terminator 2,NaN,NaN,-0.97,0.39,-0.97,1.00,1.00,NaN,NaN,-1.0
The Matrix,NaN,NaN,0.65,-0.50,0.82,1.00,1.00,NaN,NaN,NaN
The Notebook,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
Titanic,NaN,-1.0,NaN,-1.00,NaN,NaN,NaN,NaN,1.0,NaN


For one `target_item`, the next cell sorts its similarity column so students can inspect which movies are most related based on shared user ratings.


In [3]:
target_item = "Independence Day"
item_sim[target_item].drop(target_item).sort_values(ascending=False).round(3)


title
Star Wars        0.938
The Matrix       0.655
Jurassic Park   -0.106
Terminator 2    -0.972
Blade Runner       NaN
Finding Nemo       NaN
The Notebook       NaN
Titanic            NaN
Toy Story          NaN
Name: Independence Day, dtype: float64

## Technique: item-based rating prediction

To predict a user's missing rating, use similar items the same user has already rated.

- `matrix.loc[target_user].dropna()` finds the target user's known ratings.
- The evidence table keeps rated items, the user's rating for each item, and the item's similarity to the target item.
- `positive_only=True` skips negative similarities in the beginner version because they indicate opposite rating patterns.
- The prediction is a similarity-weighted average of ratings on related items.


In [4]:
def predict_item_item(matrix, target_user, target_item, k_neighbors=3, positive_only=True):
    rated = matrix.loc[target_user].dropna()
    candidates = []
    for item, rating in rated.items():
        sim = item_sim.loc[target_item, item]
        if pd.isna(sim):
            continue
        if positive_only and sim <= 0:
            continue
        candidates.append({"rated_item": item, "rating": rating, "similarity": sim})
    evidence = pd.DataFrame(candidates, columns=["rated_item", "rating", "similarity"])
    evidence = evidence.sort_values("similarity", ascending=False).head(k_neighbors)
    if evidence.empty:
        return np.nan, evidence
    numerator = (evidence["rating"] * evidence["similarity"]).sum()
    denominator = evidence["similarity"].abs().sum()
    return numerator / denominator if denominator else np.nan, evidence

target_user = "Karen"
k_neighbors = 3

pred, evidence = predict_item_item(rating_matrix, target_user, target_item, k_neighbors=k_neighbors)
print(f"Predicted Karen rating for Independence Day: {pred:.2f}")
evidence.round(3)


Predicted Karen rating for Independence Day: 6.59


,rated_item,rating,similarity
0,Star Wars,7.0,0.938
1,The Matrix,6.0,0.655


The recommender applies the same item-item prediction to every unseen movie and returns the highest predicted ratings. This is the item-item version of a top-N recommendation list.


In [5]:
def recommend_item_item(matrix, target_user, n=5, k_neighbors=3):
    unseen_items = matrix.columns[matrix.loc[target_user].isna()]
    rows = []
    for item in unseen_items:
        pred, evidence = predict_item_item(matrix, target_user, item, k_neighbors=k_neighbors, positive_only=True)
        if not pd.isna(pred):
            rows.append({
                "user": target_user,
                "recommended_movie": item,
                "predicted_rating": pred,
                "similar_rated_items": ", ".join(evidence["rated_item"].tolist()),
            })
    return pd.DataFrame(rows).sort_values("predicted_rating", ascending=False).head(n)

recommend_item_item(rating_matrix, "Karen").round(2)


,user,recommended_movie,predicted_rating,similar_rated_items
2,Karen,Toy Story,7.00,Star Wars
1,Karen,Independence Day,6.59,"Star Wars, The Matrix"
0,Karen,Finding Nemo,4.00,Jurassic Park


# Challenges

### Challenge 1 — Change the Target Item

**Goal:**
Investigate how item-item evidence changes when the target item changes.

**What to do:**

1. Change `target_item` from `"Independence Day"` to another unseen item for Karen.
2. Rerun the item similarity and prediction cells.
3. Inspect the `evidence` table.
4. Compare the similar rated items with the original result.


In [6]:
# Challenge 1
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Which rated items supported the new prediction? Were the similarities based on enough shared users? Did the predicted rating increase or decrease?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 2 — Include Negative Item Similarities

**Goal:**
Investigate how negative item similarity can affect item-item predictions.

**What to do:**

1. In `predict_item_item`, call the function with `positive_only=False`.
2. Rerun the prediction for the same `target_user` and `target_item`.
3. Compare the evidence table with the positive-only version.
4. Explain whether the negative similarities made the result easier or harder to interpret.


In [7]:
# Challenge 2
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Did any negative similarities appear? How did the predicted rating change? Would you keep or exclude negative similarities for a beginner recommender?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 3 — Concept Check: Item-Item Scalability

This challenge requires **no programming**.

Many platforms have millions of users but a smaller and more stable catalog of items.

Explain why item-item collaborative filtering can be easier to cache and serve than comparing a target user with every other user.

### Your explanation

> ................................................................................
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
